# Single-Pair EMD Benchmark — Python

Benchmarks single EMD computation using EnergyFlow (POT) and wasserstein libraries.

In [2]:
import numpy as np
import timeit
import os
from energyflow.emd import emd_pot as ef_emd
import wasserstein

def load_event(path):
    return np.loadtxt(path, delimiter=',')

sizes = [(2,2),(10,10),(50,50),(100,100),(200,200),(500,500),(1000,1000),(2000,2000),(3000,3000),
         (100,200),(200,500),(500,1000),(500,2000),(1000,2000)]

In [6]:
# Each row: (backend_label, setup_label, run_fn(ev0, ev1, w0, c0, w1, c1))
results = []
for n0, n1 in sizes:
    ev0 = load_event(os.path.join('..', 'data', f'event0_n{n0}.csv'))
    ev1 = load_event(os.path.join('..', 'data', f'event1_n{n1}.csv'))
    w0, c0 = ev0[:,0], ev0[:,1:]
    w1, c1 = ev1[:,0], ev1[:,1:]
    nrep = max(10, min(1000, int(1e6 / max(1, n0*n1))))

    # EnergyFlow POT — Euclidean only
    for norm in [True, False]:
        backend = "EF_POT"
        setup_name = f"Euclidean_{'norm' if norm else 'unnorm'}"
        ef_emd(ev0, ev1, R=1.0, beta=1.0, norm=norm)  # warmup
        t = timeit.timeit(lambda: ef_emd(ev0, ev1, R=1.0, beta=1.0, norm=norm), number=nrep) / nrep
        results.append(dict(n0=n0, n1=n1, setup=setup_name, backend=backend, time_us=t*1e6))
        print(f"{n0:5d}x{n1:<5d} {setup_name:<20s} {backend:<8s} {t*1e6:10.1f} \u00b5s")

    # wasserstein — Euclidean
    for norm in [True, False]:
        backend = "Wass"
        setup_name = f"Euclidean_{'norm' if norm else 'unnorm'}"
        emd_obj = wasserstein.EMD(1.0, 1.0, norm)
        emd_obj(w0, c0, w1, c1)  # warmup
        t = timeit.timeit(lambda: emd_obj(w0, c0, w1, c1), number=nrep) / nrep
        results.append(dict(n0=n0, n1=n1, setup=setup_name, backend=backend, time_us=t*1e6))
        print(f"{n0:5d}x{n1:<5d} {setup_name:<20s} {backend:<8s} {t*1e6:10.1f} \u00b5s")

    # wasserstein — YPhi norm=true (wasserstein's default IS YPhiArrayDistance)
    backend = "Wass"
    setup_name = "YPhi_norm"
    emd_obj = wasserstein.EMDYPhi(1.0, 1.0, True)
    emd_obj(w0, c0, w1, c1)  # warmup
    t = timeit.timeit(lambda: emd_obj(w0, c0, w1, c1), number=nrep) / nrep
    results.append(dict(n0=n0, n1=n1, setup=setup_name, backend=backend, time_us=t*1e6))
    print(f"{n0:5d}x{n1:<5d} {setup_name:<20s} {backend:<8s} {t*1e6:10.1f} \u00b5s")

    2x2     Euclidean_norm       EF_POT        944.7 µs
    2x2     Euclidean_unnorm     EF_POT        995.3 µs
    2x2     Euclidean_norm       Wass            1.4 µs
    2x2     Euclidean_unnorm     Wass            1.3 µs
    2x2     YPhi_norm            Wass            1.4 µs
   10x10    Euclidean_norm       EF_POT        963.3 µs
   10x10    Euclidean_unnorm     EF_POT        986.2 µs
   10x10    Euclidean_norm       Wass            4.9 µs
   10x10    Euclidean_unnorm     Wass            3.8 µs
   10x10    YPhi_norm            Wass            4.2 µs
   50x50    Euclidean_norm       EF_POT       1073.8 µs
   50x50    Euclidean_unnorm     EF_POT       1135.0 µs
   50x50    Euclidean_norm       Wass           95.1 µs
   50x50    Euclidean_unnorm     Wass           77.2 µs
   50x50    YPhi_norm            Wass          101.0 µs
  100x100   Euclidean_norm       EF_POT       1432.1 µs
  100x100   Euclidean_unnorm     EF_POT       1482.7 µs
  100x100   Euclidean_norm       Wass          2

In [7]:
# Save results
os.makedirs('result', exist_ok=True)
with open('result/single_emd_python.md', 'w') as f:
    f.write("# Single EMD Benchmark \u2014 Python\n\n")
    f.write("| n0 | n1 | Setup | Backend | Time (\u00b5s) |\n")
    f.write("|---|---|---|---|---|\n")
    for r in results:
        f.write(f"| {r['n0']} | {r['n1']} | {r['setup']} | {r['backend']} | {r['time_us']:.1f} |\n")
print("Results saved to result/single_emd_python.md")

Results saved to result/single_emd_python.md
